# Deploy the model to SageMaker Inference

In [1]:
import logging
import boto3
from botocore.exceptions import ClientError
import os
from pathlib import Path
# from sagemaker import image_uris
# from sagemaker.model import Model
import datetime
import numpy as np
import pandas as pd
import json

import fusion
print("Fusion loaded from:", fusion.__file__)

import importlib
import fusion
importlib.reload(fusion)

from fusion import FusionAnalyzer

np.random.seed(2)

from PIL import Image
from pylab import *
from ela import convert_to_ela_image  # centralized ELA module
from grad_cam import GradCAMExplainer  # Grad-CAM explainability (post-hoc)
from patch_localization import PatchLocalizer  # patch-level localization (post-hoc)
from mc_dropout import MCDropoutUncertainty  # MC Dropout uncertainty (post-hoc, no retraining)
from ocr import OCRExtractor  # OCR text extraction (independent modality)
from fusion import FusionAnalyzer  # cross-modal forensic fusion (rule-based)

Fusion loaded from: e:\document-tampering-detection\fusion.py


### Initialize variables

In [2]:
# aws_region='ca-central-1'
# s3_bucket = 'tamperingdetection' + datetime.datetime.now(datetime.timezone.utc).strftime("%d-%m-%Y-%H%M%S")
# model_s3_key = 'model.tar.gz'
# framework='tensorflow'
# version = '2.6'
# initial_instance_count=1
# instance_type='ml.m5.xlarge'
# py_version='py38'
# image_scope='inference'
models_dir = 'model'
models_artifact = '1'
# path_to_archive_content = os.path.join(models_dir, models_artifact)
# s3_client = boto3.client('s3')

### Create S3 bucket

In [ ]:
s3_client.create_bucket(
    Bucket=s3_bucket,
    CreateBucketConfiguration={
        'LocationConstraint': aws_region
    }
)

### Create model artifact archive

In [ ]:
!tar -czvf model.tar.gz model/1

Create a tar.gz file from the model artifacts. We have saved the model artifacts as a directory named “1” containing serialized signatures and the state needed to run them, including variable values and vocabularies to deploy to Amazon SageMaker runtime. You can also include a custom inference file inference.py within the 'code' folder in the model artifact. The custom inference can be used for pre / post processing of the input image

### Upload the model artifact to S3

In [ ]:
def upload_file(file_name, bucket, object_name=None):

    # If S3 object_name was not specified, use file_name
    if object_name is None:
        object_name = os.path.basename(file_name)

    # Upload the file
    try:
        response = s3_client.upload_file(file_name, bucket, object_name)
    except ClientError as e:
        logging.error(e)
        return False
    return True

is_uploaded = upload_file(model_s3_key, s3_bucket, None)

### Create the SageMaker Inference Endpoint

In [ ]:
# Retrieve SageMaker execution role (IAM role with permissions to access SageMaker and S3)
from sagemaker import get_execution_role
sagemaker_role = get_execution_role()

container = image_uris.retrieve(region=aws_region, 
                                framework=framework, 
                                version=version,
                                image_scope=image_scope,
                                py_version=py_version,
                                instance_type =instance_type)

model_url = f's3://{s3_bucket}/{model_s3_key}'

model = Model(image_uri=container, 
              model_data=model_url, 
              role=sagemaker_role)

endpoint_name = f"tamperingdetection-{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d-%H%M}"
print("EndpointName =", endpoint_name)

model.deploy(
    initial_instance_count=initial_instance_count,
    instance_type=instance_type,
    endpoint_name=endpoint_name
)

### Test the Inference endpoint

In [3]:
from typing import Any
from typing import Optional

# Explainability and uncertainty helpers: all post-hoc, model weights unchanged.
gradcam_model: Optional[Any] = None  # cache loaded model to avoid reloading
gradcam_explainer: Optional[GradCAMExplainer] = None  # cache Grad-CAM explainer
patch_localizer: Optional[PatchLocalizer] = None  # cache patch localizer
mc_dropout_estimator: Optional[MCDropoutUncertainty] = None  # cache MC Dropout uncertainty estimator
ocr_extractor: Optional[OCRExtractor] = None  # cache OCR extractor
fusion_analyzer: Optional[FusionAnalyzer] = None  # cache cross-modal fusion analyzer
gradcam_output_dir = Path("gradcam_outputs")  # Grad-CAM visualizations
gradcam_output_dir.mkdir(exist_ok=True)
patch_output_dir = Path("patch_outputs")  # patch localization heatmaps
patch_output_dir.mkdir(exist_ok=True)
uncertainty_output_dir = Path("uncertainty_outputs")  # uncertainty estimates and decision flags
uncertainty_output_dir.mkdir(exist_ok=True)
ocr_output_dir = Path("ocr_outputs")  # OCR extraction (text, boxes, confidence)
ocr_output_dir.mkdir(exist_ok=True)
fusion_output_dir = Path("fusion_outputs")  # cross-modal fusion results
fusion_output_dir.mkdir(exist_ok=True)

def _ensure_explainability_ready():
    # Load model and initialize all explainers once; weights/architecture unchanged.
    global gradcam_model, gradcam_explainer, patch_localizer, mc_dropout_estimator, ocr_extractor, fusion_analyzer
    if gradcam_model is None:
        import tensorflow as tf
        gradcam_model = tf.keras.models.load_model(os.path.join(models_dir, models_artifact))
        # Type narrowing: assert model loaded successfully before passing to explainers
        assert gradcam_model is not None
        gradcam_explainer = GradCAMExplainer(gradcam_model)
        patch_localizer = PatchLocalizer(gradcam_model, patch_size=64, stride=32)
        mc_dropout_estimator = MCDropoutUncertainty(gradcam_model, num_samples=30, uncertainty_threshold=0.15)
    if ocr_extractor is None:
        ocr_extractor = OCRExtractor(confidence_threshold=40)
    if fusion_analyzer is None:
        fusion_analyzer = FusionAnalyzer(
            visual_threshold=0.5,
            ocr_confidence_threshold=60,
            uncertainty_threshold=0.15,
            overlap_iou_threshold=0.3
        )

def generate_and_save_gradcam(image_path, predictions_array):
    # Generate Grad-CAM heatmap and save overlay without changing model output.
    _ensure_explainability_ready()
    # Type narrowing: assert non-None after initialization
    assert gradcam_explainer is not None
    
    # Prepare input: load original image and convert to ELA
    ela_img = convert_to_ela_image(image_path, 90).convert("RGB")
    arr = np.array(ela_img.resize((128, 128))).astype(np.float32) / 255.0
    x = arr.reshape(1, 128, 128, 3)
    
    # Compute heatmap using gradients from final conv layer
    heatmap = gradcam_explainer.make_heatmap(x)
    
    # Load original image and overlay heatmap
    original = Image.open(image_path).convert("RGB")
    overlay = gradcam_explainer.overlay_heatmap(original, heatmap, alpha=0.4)
    
    # Save visualization to disk
    output_path = gradcam_output_dir / f"{Path(image_path).stem}_gradcam.png"
    overlay.save(output_path)
    original.close()
    ela_img.close()
    
    return heatmap  # return for fusion

def generate_and_save_patch_localization(image_path):
    # Generate patch-level localization heatmap without changing model predictions.
    _ensure_explainability_ready()
    # Type narrowing: assert non-None after initialization
    assert patch_localizer is not None
    
    # Load and prepare full-resolution ELA image
    ela_img = convert_to_ela_image(image_path, 90).convert("RGB")
    arr = np.array(ela_img).astype(np.float32) / 255.0
    
    # Score each patch and aggregate into spatial heatmap
    heatmap = patch_localizer.localize(arr)
    
    # Load original and overlay patch-level scores
    original = Image.open(image_path).convert("RGB")
    overlay = patch_localizer.overlay_heatmap(original, heatmap, alpha=0.4)
    
    # Save visualization to disk
    output_path = patch_output_dir / f"{Path(image_path).stem}_patch_localization.png"
    overlay.save(output_path)
    original.close()
    ela_img.close()
    
    return heatmap  # return for fusion

def estimate_and_save_uncertainty(image_path):
    # Estimate prediction uncertainty via MC Dropout without changing single-pass prediction.
    _ensure_explainability_ready()
    # Type narrowing: assert non-None after initialization
    assert mc_dropout_estimator is not None
    
    # Prepare input: load original image and convert to ELA
    ela_img = convert_to_ela_image(image_path, 90).convert("RGB")
    arr = np.array(ela_img.resize((128, 128))).astype(np.float32) / 255.0
    x = arr.reshape(1, 128, 128, 3)
    
    # Run MC Dropout: multiple stochastic forward passes approximate posterior predictive
    mean_prob, uncertainty, flag_manual = mc_dropout_estimator.estimate(x)
    
    # Save uncertainty estimates to JSON
    output_path = uncertainty_output_dir / f"{Path(image_path).stem}_uncertainty.json"
    uncertainty_data = {
        "mc_dropout_samples": mc_dropout_estimator.num_samples,
        "mean_probability": float(mean_prob),
        "uncertainty_std": float(uncertainty),
        "uncertainty_threshold": mc_dropout_estimator.uncertainty_threshold,
        "manual_review_required": bool(flag_manual),
        "reason": "uncertainty exceeds threshold" if flag_manual else "confidence acceptable"
    }
    with open(output_path, "w") as f:
        json.dump(uncertainty_data, f, indent=2)
    
    ela_img.close()
    return mean_prob, uncertainty, flag_manual, uncertainty_data  # return data for fusion

def extract_and_save_ocr(image_path):
    # Extract text and bounding boxes via OCR; independent modality kept separate from vision predictions.
    # Rationale: vision detects pixel-level anomalies (forged pixels), OCR detects content-level info (text).
    # These are orthogonal: vision finds WHERE image was altered, OCR finds WHAT text exists.
    # Keeping separate allows independent validation and easier debugging; fusion rules can be added later.
    _ensure_explainability_ready()
    # Type narrowing: assert non-None after initialization
    assert ocr_extractor is not None
    
    # Load original image (OCR needs color/contrast, not ELA preprocessing)
    original = Image.open(image_path).convert("RGB")
    
    # Extract text tokens, bounding boxes, and confidence scores
    ocr_results = ocr_extractor.extract(original)
    
    # Generate visualization with bounding boxes overlaid on original
    viz = ocr_extractor.visualize_boxes(original, ocr_results, box_color="green", width=2)
    
    # Save OCR results as structured JSON (text + spatial + confidence for downstream processing)
    output_json = ocr_output_dir / f"{Path(image_path).stem}_ocr.json"
    ocr_extractor.save_results(ocr_results, str(output_json))
    
    # Save visualization showing detected text regions
    output_viz = ocr_output_dir / f"{Path(image_path).stem}_ocr_boxes.png"
    viz.save(output_viz)
    
    original.close()
    viz.close()
    return ocr_results

def fuse_and_save_results(image_path, gradcam_heatmap, patch_heatmap, ocr_results, uncertainty_data):
    # Perform cross-modal fusion to combine vision (Grad-CAM, patch), OCR, and uncertainty into unified risk assessment.
    # Rule-based fusion provides interpretability: each rule maps to forensic reasoning (e.g., "text altered in region X").
    # Does NOT change base prediction or endpoint behavior; fusion is post-hoc analysis for decision support.
    _ensure_explainability_ready()
    # Type narrowing: assert non-None after initialization
    assert fusion_analyzer is not None
    
    # Run rule-based fusion analyzer
    fusion_result = fusion_analyzer.analyze(
        gradcam_heatmap=gradcam_heatmap,
        patch_heatmap=patch_heatmap,
        ocr_results=ocr_results,
        uncertainty_data=uncertainty_data
    )
    
    # Save fusion results to JSON
    output_path = fusion_output_dir / f"{Path(image_path).stem}_fusion.json"
    fusion_analyzer.save_results(fusion_result, str(output_path))
    
    return fusion_result

In [4]:
# Initialize SageMaker runtime client for invoking the endpoint
# sagemaker_runtime = boto3.client("sagemaker-runtime", region_name=aws_region)

# endpoint_name = endpoint_name  # endpoint_name defined in model deployment cell

def check_image(image):
    # Single-pass inference to SageMaker endpoint; explainability/uncertainty/fusion are side effects.
    X = []
    X.append(array(convert_to_ela_image(image, 90).resize((128, 128))).flatten() / 255.0)
    X = np.array(X)

    X = X.reshape(-1, 128, 128, 3)
    
    data = {'instances': X.tolist()}
    
    # Call SageMaker endpoint: single forward pass, standard prediction
    # response = sagemaker_runtime.invoke_endpoint(
    #     EndpointName=endpoint_name, 
    #     Body=json.dumps(data),
    #     ContentType="application/json"
    #     )

    # predictions_res = json.loads(response['Body'].read().decode('utf-8'))
    # predictions = predictions_res['predictions']

    _ensure_explainability_ready()
    assert gradcam_model is not None

    predictions = gradcam_model.predict(X)
    
    print("Endpoint prediction:")
    print(predictions)
    pred_classes = np.argmax(predictions, axis=1) 
    print("Class:", pred_classes)
    
    # Generate and save Grad-CAM visualization; does not alter endpoint output
    gradcam_heatmap = generate_and_save_gradcam(image, predictions)
    
    # Generate and save patch-level localization; does not alter endpoint output
    patch_heatmap = generate_and_save_patch_localization(image)
    
    # Estimate and save MC Dropout uncertainty; does not alter endpoint output
    mc_mean, mc_uncertainty, flag_manual, uncertainty_data = estimate_and_save_uncertainty(image)
    # Type narrowing: assert non-None to allow safe attribute access
    assert mc_dropout_estimator is not None
    print(f"\nMC Dropout (N={mc_dropout_estimator.num_samples}):")
    print(f"  Mean probability: {mc_mean:.4f}")
    print(f"  Uncertainty (std): {mc_uncertainty:.4f}")
    print(f"  Manual review: {'REQUIRED' if flag_manual else 'not required'}")
    
    # Extract and save OCR results as independent modality; does not alter endpoint output
    ocr_results = extract_and_save_ocr(image)
    print(f"\nOCR Extraction (Independent Modality):")
    print(f"  Tokens detected: {len(ocr_results)}")
    if ocr_results:
        avg_confidence = np.mean([r['confidence'] for r in ocr_results])
        print(f"  Mean confidence: {avg_confidence:.1f}%")
        print(f"  Sample text: {' '.join([r['text'][:20] for r in ocr_results[:3]])}...")
    
    # Cross-modal fusion: combine all modalities into unified risk assessment; does not alter endpoint output
    fusion_result = fuse_and_save_results(image, gradcam_heatmap, patch_heatmap, ocr_results, uncertainty_data)
    print(f"\n{'='*60}")
    print(f"FORENSIC FUSION ANALYSIS")
    print(f"{'='*60}")
    print(f"Risk Score: {fusion_result['risk_score']:.3f} / 1.0")
    print(f"Risk Tier: {fusion_result['risk_tier']}")
    print(f"\nExplanations:")
    for i, explanation in enumerate(fusion_result['explanations'], 1):
        print(f"  {i}. {explanation}")
    print(f"\nDetailed Scores:")
    for component, score in fusion_result['details'].items():
        print(f"  - {component}: {score:.3f}")
    print(f"{'='*60}")

In [5]:
check_image('images/predict/Paystub.jpg')

1/1 [==============================] - 0s 162ms/step
Endpoint prediction:
[[0.6943598 0.3056402]]
Class: [0]

MC Dropout (N=30):
  Mean probability: 0.3068
  Uncertainty (std): 0.0516
  Manual review: not required

OCR Extraction (Independent Modality):
  Tokens detected: 194
  Mean confidence: 93.5%
  Sample text: CLOCK VCHR NO,...

FORENSIC FUSION ANALYSIS
Risk Score: 0.567 / 1.0
Risk Tier: MEDIUM

Explanations:
  1. Visual anomaly detected: 0.65 exceeds threshold 0.5
  2. Visual tamper overlaps 194 OCR text regions (mean IoU: 1.00); possible text manipulation
  3. High OCR confidence in visually suspicious regions: 0.65; text may be cleanly forged (sophisticated tampering)

Detailed Scores:
  - visual_anomaly: 0.647
  - visual_ocr_overlap: 1.000
  - ocr_visual_conflict: 0.651
  - low_ocr_confidence: 0.000
  - uncertainty_penalty: 0.000
  - spatial_density: 0.073


In [6]:
check_image('images/predict/TamperedPaystub.jpg')

1/1 [==============================] - 0s 29ms/step
Endpoint prediction:
[[0.5553695 0.4446305]]
Class: [0]

MC Dropout (N=30):
  Mean probability: 0.4609
  Uncertainty (std): 0.0710
  Manual review: not required

OCR Extraction (Independent Modality):
  Tokens detected: 187
  Mean confidence: 90.7%
  Sample text: co, FILE CLOCK...

FORENSIC FUSION ANALYSIS
Risk Score: 0.497 / 1.0
Risk Tier: MEDIUM

Explanations:
  1. Visual anomaly detected: 0.51 exceeds threshold 0.5
  2. Visual tamper overlaps 169 OCR text regions (mean IoU: 0.94); possible text manipulation
  3. High OCR confidence in visually suspicious regions: 0.65; text may be cleanly forged (sophisticated tampering)

Detailed Scores:
  - visual_anomaly: 0.511
  - visual_ocr_overlap: 0.904
  - ocr_visual_conflict: 0.647
  - low_ocr_confidence: 0.000
  - uncertainty_penalty: 0.000
  - spatial_density: 0.008


### Cleanup

In [ ]:
# delete sagemaker inference endpoint
client = boto3.client('sagemaker')
client.delete_endpoint(
    EndpointName=endpoint_name
)


In [ ]:
# empty s3 bucket and delete the bucket
s3_resource = boto3.resource('s3')
bucket = s3_resource.Bucket(s3_bucket)
bucket.objects.all().delete()
bucket.delete()
print("S3 bucket and contents deleted")